# **Exploratory Data Analysis (EDA) of Google Play Store Dataset**

# The Google Play Store dataset contains information about thousands of mobile applications available on the Google Play Store. It includes various attributes such as app name, category, rating, number of reviews, size, installs, price, content rating, and last update. This dataset helps analyze trends in app performance, user engagement, and market behavior. Through exploratory data analysis EDA, we can identify patterns such as the most popular app categories, the relationship between ratings and installs, pricing strategies, and other factors that influence an app’s success in the marketplace.

# step-1 import libraries

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# step-2 Load dataset

In [ ]:
df = pd.read_csv('googleplaystore.csv')

# To show top 5 rows

In [ ]:
df.head()

# to show rows and columns

In [ ]:
df.shape

# to check Data types

In [ ]:
df.info()

# Statistical summary

In [ ]:
df.describe().T

# to show  Duplicate rows

In [ ]:
df.duplicated().sum()

# Data Cleaning

# Check missing counts

In [ ]:
df.isnull().sum()

# Drop rows where Rating is missing

In [ ]:
df = df.dropna(subset=['Rating']).reset_index(drop=True)

# Fill missing categorical values

In [ ]:
df['Type'].fillna('Free', inplace=True)

In [ ]:
# Fill 'Content Rating' with 'Everyone'
df['Content Rating'].fillna('Everyone', inplace=True)

In [ ]:
# Fill 'Android Ver' with mode
df['Android Ver'].fillna(df['Android Ver'].mode()[0], inplace=True)

In [ ]:
# Fill 'Current Ver' with 'Varies with device'
df['Current Ver'].fillna('Varies with device', inplace=True)

# Convert data types

In [ ]:
# Reviews to numeric
df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')

# Feature Engineering

In [ ]:
# Extract year from Last Updated
df['Update Year'] = df['Last Updated'].dt.year

In [ ]:
# Create install bins
bins = [0, 1000, 10000, 100000, 1000000, 1e9]
labels = ['<1K', '1K-10K', '10K-100K', '100K-1M', '>1M']
df['Installs Category'] = pd.cut(df['Installs'], bins=bins, labels=labels)

In [ ]:
# Rating groups
df['Rating Group'] = pd.cut(df['Rating'], bins=[0,2,3,4,5], labels=['Poor','Average','Good','Excellent'])

In [ ]:
# Free vs Paid flag
df['Is Free'] = df['Price'] == 0

In [ ]:
# Log transforms for skewed variables (for analysis)
df['Log Reviews'] = np.log1p(df['Reviews'])
df['Log Installs'] = np.log1p(df['Installs'])

# Exploratory Data Analysis

# Distribution of App Ratings

In [ ]:
fig = px.histogram(df, x='Rating', nbins=40, title='📈 Distribution of App Ratings',
                   labels={'Rating':'Rating'}, marginal='box')
fig.show()

# Distribution of Reviews

In [ ]:
fig = px.histogram(df, x='Log Reviews', nbins=40, title='📊 Distribution of Reviews (log)',
                   labels={'Log Reviews':'Log(Reviews+1)'}, marginal='violin')
fig.show()

# Distribution of Installs

In [ ]:
fig = px.histogram(df, x='Log Installs', nbins=40, title='📦 Distribution of Installs (log)',
                   labels={'Log Installs':'Log(Installs+1)'}, marginal='box')
fig.show()

# Top 10 Categories by Number of Apps

In [ ]:
top_cats = df['Category'].value_counts().head(10).reset_index()
top_cats.columns = ['Category', 'Count']
fig = px.bar(top_cats, x='Count', y='Category', orientation='h',
             title=' Top 10 App Categories', color='Count', color_continuous_scale='viridis')
fig.show()

# Content Rating Distribution (Pie Chart)

In [ ]:
content_counts = df['Content Rating'].value_counts().reset_index()
content_counts.columns = ['Content Rating', 'Count']
fig = px.pie(content_counts, values='Count', names='Content Rating',
             title=' Content Rating Distribution', hole=0.3)
fig.show()

# Scatter Plot  Ratings vs Installs

In [ ]:
sample = df.sample(min(5000, len(df)), random_state=42)
fig = px.scatter(sample, x='Rating', y='Installs', color='Category',
                 title=' Ratings vs Installs (log scale)', log_y=True,
                 hover_data=['App'], opacity=0.6)
fig.show()

# Free vs Paid Apps (Pie)

In [ ]:
free_paid_counts = df['Is Free'].value_counts().reset_index()
free_paid_counts.columns = ['Is Free', 'Count']
free_paid_counts['Type'] = free_paid_counts['Is Free'].map({True:'Free', False:'Paid'})
fig = px.pie(free_paid_counts, values='Count', names='Type',
             title='Free vs Paid Apps', hole=0.4)
fig.show()

# Heatmap  Correlation of Numerical Features

In [ ]:
num_cols = ['Rating', 'Reviews', 'Installs', 'Price']
corr = df[num_cols].corr()
fig = px.imshow(corr, text_auto=True, title='Correlation Heatmap',
                color_continuous_scale='RdBu', zmin=-1, zmax=1)
fig.show()

# Line Chart Number of Apps Updated per Year

In [ ]:
yearly = df['Update Year'].value_counts().sort_index().reset_index()
yearly.columns = ['Year', 'Count']
fig = px.line(yearly, x='Year', y='Count', markers=True,
              title='App Updates Over Time')
fig.show()

# Violin Plot Ratings by Content Rating

In [ ]:
fig = px.violin(df, x='Content Rating', y='Rating', box=True,
                title=' Rating Distribution by Content Rating')
fig.show()

# 3D Scatter Rating, Log Installs, Log Reviews

In [ ]:
fig = px.scatter_3d(df.sample(2000), x='Rating', y='Log Installs', z='Log Reviews',
                    color='Category', title=' 3D Scatter: Rating, Installs, Reviews')
fig.show()

# Sunburst Chart – Category → Content Rating

In [ ]:
fig = px.sunburst(df, path=['Category', 'Content Rating'], values='Reviews',
                  title=' Sunburst: Category & Content Rating (by total reviews)')
fig.show()

# Parallel Categories – Installs Category × Content Rating × Type

In [ ]:
fig = px.parallel_categories(df[['Installs Category', 'Content Rating', 'Is Free']].dropna(),
                             title='Parallel Categories: Installs × Content × Free/Paid')
fig.show()

In [ ]:
print("""
📌 KEY INSIGHTS FROM EDA:

1. **Market Landscape:**
   - 92% of apps are free.
   - FAMILY and GAME categories have the most apps.
   - Most apps target 'Everyone' (80%+).

2. **Ratings:**
   - Average rating ~4.2; most apps rated between 4.0-4.5.
   - Events, Education, Art & Design have highest average ratings.
   - No strong correlation between price and rating.

3. **Installs & Reviews:**
   - Reviews and installs are strongly correlated.
   - Communication and Social apps have the highest installs.
   - A few apps (Facebook, WhatsApp) dominate reviews and installs.

4. **Pricing:**
   - Paid apps are rare (8%) and mostly under $10.
   - Finance and Medical apps are the most expensive.

5. **Trends:**
   - App updates peaked around 2018.
   - Older Android versions still required by many apps.

6. **Outliers:**
   - Some categories (e.g., Medical) contain very high-priced apps.
   - Unrated apps have significantly lower installs.
""")

# Save Cleaned Dataset

In [ ]:
df.to_csv('googleplaystore_cleaned.csv', index=False)
print("✅ Cleaned data saved as 'googleplaystore_cleaned.csv'")